## 2026 MIAMI GRAND PRIX — PODIUM PREDICTION
### Rolling data: Australia (R1) + China (R2) + Japan (R3), Miami qualifying grid + weather sourced live (May 2026) - update to readme


In [19]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
 
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

### Pulling requests from Fast F1 API

In [ ]:
import sys
print(sys.executable)

/usr/bin/python3


In [ ]:
import sys
!{sys.executable} -m pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00


In [ ]:
import fastf1
import pandas as pd

import numpy as np
import os

In [ ]:
# ── Load the session ──────────────────────────────────────────────────────────
session = fastf1.get_session(2026, "Japan", "Q")
session.load(telemetry=False, weather=True, messages=False)

laps = session.laps


req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Japanese Grand Prix - Qualifying [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_statu

In [ ]:
# ── Best individual sector times per driver (ultimate lap) ────────────────────
# FastF1 stores sector times as timedeltas — convert to seconds
def td_to_s(td):
    """Convert timedelta (or NaT) to float seconds."""
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()

records = []
for driver in laps["Driver"].unique():
    drv_all = laps.pick_drivers(driver).pick_wo_box()
    
    if drv_all.empty:
        continue  # skip drivers with no valid laps

    best_s1 = drv_all["Sector1Time"].dropna().min()
    best_s2 = drv_all["Sector2Time"].dropna().min()
    best_s3 = drv_all["Sector3Time"].dropna().min()
    best_lap = drv_all["LapTime"].dropna().min()  # ← get best lap directly

    records.append({
        "Driver":    driver,
        "S1":        td_to_s(best_s1),
        "S2":        td_to_s(best_s2),
        "S3":        td_to_s(best_s3),
        "BestLap_s": td_to_s(best_lap),
    })

quali_df = pd.DataFrame(records)

# ── Ultimate lap = S1 + S2 + S3 ───────────────────────────────────────────────
quali_df["UltimateLap_s"] = (
    quali_df["S1"] + quali_df["S2"] + quali_df["S3"]
).round(3)

# ── Gap from pole (use best Q3 / overall best lap) ───────────────────────────
pole_time = quali_df["BestLap_s"].min()
quali_df["JapanGapFromPole_s"] = (quali_df["BestLap_s"] - pole_time).round(3)

# Percentage gap — more comparable across circuits than raw seconds
quali_df["JapanGapFromPolePct"] = (
    quali_df["JapanGapFromPole_s"] / pole_time * 100
).round(4)

# ── Grid position (sort by best lap) ─────────────────────────────────────────
quali_df = quali_df.sort_values("BestLap_s").reset_index(drop=True)
quali_df["JapanGrid"] = quali_df.index + 1

# Adding The tremperature + rainfall probability on that day
# Japan GP weather — sourced manually (e.g. Weather.com / AccuWeather)
rain_probability = 0.10   # replace with actual figure
temperature      = 18.0   # replace with actual figure

quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

# Adding the Driver points from the 2 races(Australia and Chinese GPs)
driver_points = {
    "RUS": 44, "LEC": 38, "ANT": 35, "NOR": 32, "HAM": 28,
    "PIA": 22, "VER": 18, "BEA": 14, "LIN": 12, "BOR": 10,
    "GAS": 8,  "LAW": 6,  "OCO": 4,  "ALB": 4,  "HAD": 2,
    "COL": 1,  "SAI": 0,  "ALO": 0,  "BOT": 0,
    "STR": 0,  "PER": 0,  "HUL": 0,
}

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)


print(quali_df[["Driver", "S1", "S2", "S3", "UltimateLap_s",
                "BestLap_s", "JapanGrid", "JapanGapFromPole_s",
                "JapanGapFromPolePct", "RainProbability", "Temperature", "DriverPointsNorm"]])

quali_df.to_csv("japan_quali.csv", index=False)
print("\nSaved → japan_quali.csv")

   Driver      S1      S2      S3  UltimateLap_s  BestLap_s  JapanGrid  \
0     ANT  31.827  39.398  17.464         88.689     88.778          1   
1     RUS  31.782  39.607  17.616         89.005     89.076          2   
2     PIA  31.954  39.557  17.577         89.088     89.132          3   
3     LEC  31.655  39.855  17.668         89.178     89.303          4   
4     NOR  32.049  39.716  17.627         89.392     89.409          5   
5     HAM  31.762  39.933  17.625         89.320     89.567          6   
6     GAS  32.143  40.029  17.519         89.691     89.691          7   
7     HAD  32.164  40.094  17.608         89.866     89.978          8   
8     BOR  32.288  40.149  17.492         89.929     89.990          9   
9     LIN  32.548  40.043  17.501         90.092     90.109         10   
10    VER  32.369  40.196  17.565         90.130     90.262         11   
11    OCO  32.481  40.048  17.700         90.229     90.309         12   
12    HUL  32.400  40.181  17.564     

## Building rolling features from AUstralia and Chinese GPs

In [ ]:
# Let us build the rolling features from the average of the Australian and ChineseGPs grid positions and final race positions
aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}
 


In [ ]:
# China finish positions (confirmed classification)
# DNF: VER=16, ALO=17, STR=18 | DNS: NOR=19, BOR=20, ALB=21, PIA=22
china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

In [ ]:
# Australia grid positions (confirmed qualifying)
# VER P20 (crashed Q1), SAI P21 & STR P22 (power unit issues in quali)
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

In [ ]:
# China grid positions (confirmed qualifying)
# ALB P18 but started from pit lane (parc fermé modification)
china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

In [ ]:
# CONSTRUCTOR STANDINGS after China (Round 2)
# Source: formula1.com / total-motorsport.com
# ─────────────────────────────────────────────────────────────────────────────
team_points = {
    "Mercedes":     98,
    "Ferrari":      67,
    "McLaren":      18,
    "Haas":         17,
    "Red Bull":     12,
    "Racing Bulls": 12,
    "Alpine":       10,
    "Audi":          2,
    "Williams":      2,
    "Cadillac":      0,
    "Aston Martin":  0,
}

In [ ]:
# DRIVER STANDINGS after China (Round 2)
# Source: autohebdof1.com / total-motorsport.com
# ─────────────────────────────────────────────────────────────────────────────
driver_points = {
    "RUS": 51, "ANT": 47, "LEC": 34, "HAM": 33,
    "BEA": 17, "NOR": 15, "GAS":  9, "VER":  8,
    "LAW":  8, "LIN":  4, "HAD":  4, "PIA":  3,
    "SAI":  2, "BOR":  2, "COL":  1, "OCO":  0,
    "HUL":  0, "ALB":  0, "BOT":  0, "PER":  0,
    "ALO":  0, "STR":  0,
}

In [ ]:
# DRIVER → TEAM MAPPING (2026 season)
# ─────────────────────────────────────────────────────────────────────────────
driver_to_team = {
    "RUS": "Mercedes",    "ANT": "Mercedes",
    "HAM": "Ferrari",     "LEC": "Ferrari",
    "NOR": "McLaren",     "PIA": "McLaren",
    "VER": "Red Bull",    "HAD": "Red Bull",
    "BEA": "Haas",        "OCO": "Haas",
    "LAW": "Racing Bulls","LIN": "Racing Bulls",
    "HUL": "Audi",        "BOR": "Audi",
    "GAS": "Alpine",      "COL": "Alpine",
    "SAI": "Williams",    "ALB": "Williams",
    "PER": "Cadillac",    "BOT": "Cadillac",
    "ALO": "Aston Martin","STR": "Aston Martin",
}

In [ ]:
# BUILD ROLLING FEATURES
# ─────────────────────────────────────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)  # 22
 
records = []
for drv in drivers:
    aus_fp  = aus_finish[drv]
    chn_fp  = china_finish[drv]
    aus_gp  = aus_grid[drv]
    chn_gp  = china_grid[drv]
 
    # Average finish position (dense rank, so DNFs are ranked last)
    avg_finish_pos = round((aus_fp + chn_fp) / 2, 2)
 
    # Normalised average finish (0 = best, 1 = worst)
    avg_finish_norm = round(((aus_fp - 1) / (N - 1) + (chn_fp - 1) / (N - 1)) / 2, 4)
 
    # Average grid position
    avg_grid_pos = round((aus_gp + chn_gp) / 2, 2)
 
    # Normalised average grid (0 = pole, 1 = last)
    avg_grid_norm = round(((aus_gp - 1) / (N - 1) + (chn_gp - 1) / (N - 1)) / 2, 4)
 
    # DNF rate (1 race out of 2 if DNF/DNS in either)
    dnf_aus = 1 if aus_fp >= 18 else 0   # positions 18-22 = DNF or DNS
    dnf_chn = 1 if chn_fp >= 16 else 0   # VER(16), ALO(17), STR(18) DNF; NOR+ DNS
    dnf_rate = round((dnf_aus + dnf_chn) / 2, 2)
 
    records.append({
        "Driver":          drv,
        "Team":            driver_to_team[drv],
        "AusFinish":       aus_fp,
        "ChinaFinish":     chn_fp,
        "AvgFinishPos":    avg_finish_pos,
        "AvgFinishNorm":   avg_finish_norm,
        "AusGrid":         aus_gp,
        "ChinaGrid_prev":  chn_gp,
        "AvgGridPos":      avg_grid_pos,
        "AvgGridNorm":     avg_grid_norm,
        "DNF_rate":        dnf_rate,
    })
 
rolling_df = pd.DataFrame(records)

In [ ]:
# NORMALISE TEAM & DRIVER STANDINGS
# ─────────────────────────────────────────────────────────────────────────────
max_team_pts   = max(team_points.values())
max_driver_pts = max(driver_points.values())
 
rolling_df["TeamScore"]       = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)
 
rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)
 

In [ ]:
# PRINT SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("  ROLLING FEATURES FOR JAPAN GP PREDICTION")
print("  Based on: Australia (R1) + China (R2)")
print("=" * 80)
print(f"\n{'DRV':<6}{'Team':<16}{'AUS':<5}{'CHN':<5}{'AvgFin':<9}"
      f"{'AvgFinNorm':<12}{'AvgGrid':<9}{'AvgGridNorm':<13}"
      f"{'DNF%':<7}{'TeamScore':<11}{'DrvPtsNorm'}")
print("─" * 100)
for _, r in rolling_df.sort_values("AvgFinishPos").iterrows():
    print(f"{r['Driver']:<6}{r['Team']:<16}{r['AusFinish']:<5}{r['ChinaFinish']:<5}"
          f"{r['AvgFinishPos']:<9}{r['AvgFinishNorm']:<12}{r['AvgGridPos']:<9}"
          f"{r['AvgGridNorm']:<13}{r['DNF_rate']:<7}{r['TeamScore']:<11}"
          f"{r['DriverPointsNorm']}")
 
rolling_df.to_csv("rolling_features_aus_china.csv", index=False)
print("\nSaved → rolling_features_aus_china.csv")
 


  ROLLING FEATURES FOR JAPAN GP PREDICTION
  Based on: Australia (R1) + China (R2)

DRV   Team            AUS  CHN  AvgFin   AvgFinNorm  AvgGrid  AvgGridNorm  DNF%   TeamScore  DrvPtsNorm
────────────────────────────────────────────────────────────────────────────────────────────────────
RUS   Mercedes        1    2    1.5      0.0238      1.5      0.0238       0.0    1.0        1.0
ANT   Mercedes        2    1    1.5      0.0238      1.5      0.0238       0.0    1.0        0.9216
HAM   Ferrari         4    3    3.5      0.119       5.0      0.1905       0.0    0.6837     0.6471
LEC   Ferrari         3    4    3.5      0.119       4.0      0.1429       0.0    0.6837     0.6667
BEA   Haas            7    5    6.0      0.2381      13.0     0.5714       0.0    0.1735     0.3333
GAS   Alpine          10   6    8.0      0.3333      10.5     0.4524       0.0    0.102      0.1765
LAW   Racing Bulls    13   7    10.0     0.4286      11.0     0.4762       0.0    0.1224     0.1569
LIN   Racing 

In [ ]:
# HOW TO MERGE INTO YOUR MAIN DATAFRAME
# ─────────────────────────────────────────────────────────────────────────────
# After building your Japan quali df, merge like this:
#
df = quali_df.merge(
    rolling_df[["Driver", "AvgFinishNorm", "AvgGridNorm",
                "DNF_rate", "TeamScore", "Team"]],
    on="Driver", how="left"
   )

In [ ]:
df.head()

,Driver,S1,S2,S3,BestLap_s,UltimateLap_s,JapanGapFromPole_s,JapanGapFromPolePct,JapanGrid,RainProbability,Temperature,DriverPoints,DriverPointsNorm,AvgFinishNorm,AvgGridNorm,DNF_rate,TeamScore,Team
0,ANT,31.827,39.398,17.464,88.778,88.689,0.000,0.0000,1,0.1,18.0,35,0.7955,0.0238,0.0238,0.0,1.0000,Mercedes
1,RUS,31.782,39.607,17.616,89.076,89.005,0.298,0.3357,2,0.1,18.0,44,1.0000,0.0238,0.0238,0.0,1.0000,Mercedes
2,PIA,31.954,39.557,17.577,89.132,89.088,0.354,0.3987,3,0.1,18.0,22,0.5000,0.9762,0.1905,1.0,0.1837,McLaren
3,LEC,31.655,39.855,17.668,89.303,89.178,0.525,0.5914,4,0.1,18.0,38,0.8636,0.1190,0.1429,0.0,0.6837,Ferrari
4,NOR,32.049,39.716,17.627,89.409,89.392,0.631,0.7108,5,0.1,18.0,32,0.7273,0.5238,0.2381,0.5,0.1837,McLaren


 ## Analysing FP2 pace for Japan

In [ ]:
# ── Load FP2 ──────────────────────────────────────────────────────────────────
session = fastf1.get_session(2026, "Japan", "FP2")
session.load(telemetry=False, weather=False, messages=False)

laps = session.laps.copy()

# ── Helper: convert timedelta to seconds ─────────────────────────────────────
def td_to_s(td):
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()

laps["LapTime_s"] = laps["LapTime"].apply(td_to_s)

# ── Filter criteria ───────────────────────────────────────────────────────────
# Keep only representative race-sim laps:
#   - Not an in/out lap (IsAccurate flag or LapNumber > 1 in stint)
#   - Lap time within 110% of session median (removes outliers, VSC laps)
#   - Compound is MEDIUM or HARD (race compounds — not quali soft runs)
#   - Stint lap number >= 3 (skip initial warm-up laps)

laps = laps[laps["IsAccurate"] == True].copy()  # FastF1 flags dodgy laps

COMPOUNDS = ["MEDIUM", "HARD"]
laps = laps[laps["Compound"].isin(COMPOUNDS)]

# Stint lap number — laps within a stint counted from 1
laps["StintLapNum"] = laps.groupby(["Driver", "Stint"]).cumcount() + 1
laps = laps[laps["StintLapNum"] >= 3]

# Remove outliers: keep within 107% of driver's own median
def within_107(group):
    med = group["LapTime_s"].median()
    return group[group["LapTime_s"] <= med * 1.07]

laps = laps.groupby("Driver", group_keys=False).apply(within_107)

# ── Aggregate per driver ──────────────────────────────────────────────────────
records = []
for driver, grp in laps.groupby("Driver"):
    med_pace   = grp["LapTime_s"].median()
    mean_pace  = grp["LapTime_s"].mean()
    best_pace  = grp["LapTime_s"].min()
    lap_count  = len(grp)

    # Compound breakdown (useful for context)
    compound_counts = grp["Compound"].value_counts().to_dict()

    records.append({
        "Driver":           driver,
        "FP2_MedianPace_s": round(med_pace, 3),
        "FP2_MeanPace_s":   round(mean_pace, 3),
        "FP2_BestPace_s":   round(best_pace, 3),
        "FP2_LapCount":     lap_count,
        "FP2_Compounds":    str(compound_counts),
    })

fp2_df = pd.DataFrame(records).sort_values("FP2_MedianPace_s").reset_index(drop=True)

# ── Gap from fastest driver in FP2 ───────────────────────────────────────────
ref_pace = fp2_df["FP2_MedianPace_s"].min()
fp2_df["FP2_GapToFastest_s"] = (fp2_df["FP2_MedianPace_s"] - ref_pace).round(3)

print(fp2_df[["Driver", "FP2_MedianPace_s", "FP2_GapToFastest_s",
              "FP2_LapCount", "FP2_Compounds"]])

fp2_df.to_csv("japan_fp2_pace.csv", index=False)
print("\nSaved → japan_fp2_pace.csv")


core           INFO 	Loading data for Japanese Grand Prix - Practice 2 [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Japanese Grand Prix - Practice 2 [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for se

   Driver  FP2_MedianPace_s  FP2_GapToFastest_s  FP2_LapCount   FP2_Compounds
0     ANT            94.496               0.000             7   {'MEDIUM': 7}
1     STR            95.042               0.546             2   {'MEDIUM': 2}
2     RUS            95.079               0.583             6   {'MEDIUM': 6}
3     LEC            95.243               0.747             8   {'MEDIUM': 8}
4     PIA            95.808               1.312             8   {'MEDIUM': 8}
5     GAS            95.919               1.423            10    {'HARD': 10}
6     OCO            95.958               1.462            12  {'MEDIUM': 12}
7     HUL            96.146               1.650             6     {'HARD': 6}
8     HAD            96.161               1.665             9     {'HARD': 9}
9     BEA            96.170               1.674            10  {'MEDIUM': 10}
10    VER            96.299               1.803            12    {'HARD': 12}
11    HAM            96.424               1.928             3   

/tmp/ipykernel_9900/275636439.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps["StintLapNum"] = laps.groupby(["Driver", "Stint"]).cumcount() + 1
/tmp/ipykernel_9900/275636439.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  laps = laps.groupby("Driver", group_keys=False).apply(within_107)


In [ ]:
df_final = df.merge(
    fp2_df[["Driver", "FP2_GapToFastest_s"]],
    on="Driver", how="left"
   )

In [ ]:
df_final.head()

,Driver,S1,S2,S3,BestLap_s,UltimateLap_s,JapanGapFromPole_s,JapanGapFromPolePct,JapanGrid,RainProbability,Temperature,DriverPoints,DriverPointsNorm,AvgFinishNorm,AvgGridNorm,DNF_rate,TeamScore,Team,FP2_GapToFastest_s
0,ANT,31.827,39.398,17.464,88.778,88.689,0.000,0.0000,1,0.1,18.0,35,0.7955,0.0238,0.0238,0.0,1.0000,Mercedes,0.000
1,RUS,31.782,39.607,17.616,89.076,89.005,0.298,0.3357,2,0.1,18.0,44,1.0000,0.0238,0.0238,0.0,1.0000,Mercedes,0.583
2,PIA,31.954,39.557,17.577,89.132,89.088,0.354,0.3987,3,0.1,18.0,22,0.5000,0.9762,0.1905,1.0,0.1837,McLaren,1.312
3,LEC,31.655,39.855,17.668,89.303,89.178,0.525,0.5914,4,0.1,18.0,38,0.8636,0.1190,0.1429,0.0,0.6837,Ferrari,0.747
4,NOR,32.049,39.716,17.627,89.409,89.392,0.631,0.7108,5,0.1,18.0,32,0.7273,0.5238,0.2381,0.5,0.1837,McLaren,17.719


## Feature Engineering

In [ ]:
feature_cols = [
"UltimateLap_s",        # Japan quali: best S1+S2+S3
"JapanGapFromPole_s",   # Japan quali: gap to pole
"JapanGrid",            # Japan qualifying position
"FP2_GapToFastest_s",   # FP2 long-run race pace
"TeamScore",            # Constructor points normalised (after China)
"DriverPointsNorm",     # Driver points normalised (after China)
"AvgFinishNorm",        # Rolling avg finish (Aus + China)
"AvgGridNorm",          # Rolling avg grid (Aus + China)
"DNF_rate",             # Reliability signal
"RainProbability",      # Weather (manual)
"Temperature",          # Weather (manual)
]

In [ ]:
# Lets build our target variable
# ── Target: rolling normalised race score from Aus + China ───────────────
n_drivers = len(df_final)

# Normalise average finish position (0 = best, 1 = worst)
avg_finish_norm = df_final["AvgFinishNorm"]  # already in your df

# Normalise average grid position (0 = best, 1 = worst)  
avg_grid_norm = df_final["AvgGridNorm"]      # already in your df

# Composite target — weight finish more than grid (60/40)
df_final["RaceScore"] = (
    0.6 * avg_finish_norm +
    0.4 * avg_grid_norm
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

In [ ]:
import sklearn
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

In [ ]:
# Filling any missing values with median
from sklearn.impute import SimpleImputer
imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

### Modelling

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# ── Compare multiple models with LOO-CV ──────────────────────────────────────
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

In [ ]:
models = {
    "Ridge":        Ridge(alpha=1.0),
    "ElasticNet":   ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest": RandomForestRegressor(
                        n_estimators=100, max_depth=3,
                        min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                        n_estimators=100, learning_rate=0.05,
                        max_depth=2, subsample=0.8, random_state=42),
    "XGBoost":      XGBRegressor(
                        n_estimators=100, learning_rate=0.05,
                        max_depth=2, subsample=0.8,
                        colsample_bytree=0.8, reg_lambda=2.0,
                        random_state=42),
}

loo = LeaveOneOut()
results = {}

for name, mdl in models.items():
    # Ridge and ElasticNet benefit from scaling
    if name in ["Ridge", "ElasticNet"]:
        pipe = Pipeline([("scaler", StandardScaler()), ("model", mdl)])
    else:
        pipe = Pipeline([("model", mdl)])

    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        X_tr, X_te = X_imputed[train_idx], X_imputed[test_idx]
        y_tr, y_te = y.iloc[train_idx],    y.iloc[test_idx]
        pipe.fit(X_tr, y_tr)
        errors.append(abs(pipe.predict(X_te)[0] - y_te.iloc[0]))

    results[name] = {
        "MAE":    round(np.mean(errors), 4),
        "StdDev": round(np.std(errors), 4),
    }


In [ ]:
# ── Print comparison ──────────────────────────────────────────────────────────
print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")


Model               LOO MAE  Std Dev  Verdict
───────────────────────────────────────────────────────
Ridge                0.0245   0.0257   ← best
GradientBoost        0.0524   0.0615  
ElasticNet           0.0584   0.0374  
XGBoost              0.0687   0.0740  
RandomForest         0.0806   0.0710  


In [ ]:
# ── Use the best model for final prediction ───────────────────────────────────
best_name = min(results, key=lambda x: results[x]["MAE"])
print(f"\nUsing: {best_name}")

best_model = models[best_name]
if best_name in ["Ridge", "ElasticNet"]:
    final_pipe = Pipeline([("scaler", StandardScaler()), ("model", best_model)])
else:
    final_pipe = Pipeline([("model", best_model)])

final_pipe.fit(X_imputed, y)
df["PredictedScore"] = final_pipe.predict(X_imputed)


Using: Ridge


In [ ]:
final_pipe.fit(X_imputed, y)
df["PredictedScore"] = final_pipe.predict(X_imputed)

# ── Predicted finishing order ─────────────────────────────────────────────────
final = df.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 68)
print("   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX")
print("   Target: Aus + China avg finish + avg grid (rolling)")
print("=" * 68)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'Score'}")
print("  " + "─" * 40)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}{row['PredictedScore']:.4f}")

podium = final.head(3)
print(f"\n  {'='*45}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:        {best_name}")
print(f"  LOO MAE:           {results[best_name]['MAE']:.4f}")
print(f"  LOO Std Dev:       {results[best_name]['StdDev']:.4f}")
print(f"  Weather:           {df['Temperature'].iloc[0]:.1f}°C | Rain: {df['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*45}")

# ── Feature importances (tree models only) ────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<28} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients (higher = more influence):")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        bar = "█" * int(abs(coef) * 20)
        print(f"  {feat:<28} {coef:+.4f}  {bar}")


   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX
   Target: Aus + China avg finish + avg grid (rolling)
  P   DRV   Team                  Score
  ────────────────────────────────────────
  1   RUS   Mercedes              0.0203
  2   ANT   Mercedes              0.0216
  3   LEC   Ferrari               0.1321
  4   HAM   Ferrari               0.1572
  5   GAS   Alpine                0.3976
  6   BEA   Haas                  0.4036
  7   NOR   McLaren               0.4130
  8   LAW   Racing Bulls          0.4638
  9   LIN   Racing Bulls          0.4646
  10  HAD   Red Bull              0.4983
  11  OCO   Haas                  0.5250
  12  VER   Red Bull              0.5311
  13  COL   Alpine                0.5479
  14  BOR   Audi                  0.6049
  15  PIA   McLaren               0.6339
  16  SAI   Williams              0.6345
  17  HUL   Audi                  0.6427
  18  ALB   Williams              0.7086
  19  PER   Cadillac              0.7512
  20  BOT   Cadillac      

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Get predictions from all models ──────────────────────────────────────────
all_predictions = {}

for name, mdl in models.items():
    if name in ["Ridge", "ElasticNet"]:
        pipe = Pipeline([("scaler", StandardScaler()), ("model", mdl)])
    else:
        pipe = Pipeline([("model", mdl)])

    pipe.fit(X_imputed, y)
    preds = pipe.predict(X_imputed)
    ranked = pd.Series(preds, index=df["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df["Driver"]
).rank(method="min").astype(int)

pred_df = pred_df.sort_values("BestModelPos")

# ── Print all model predictions side by side ──────────────────────────────────
print("\n" + "=" * 85)
print("   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX")
print("   All models compared")
print("=" * 85)

col_w = 14
header = f"  {'DRV':<6}{'Team':<20}"
for name in models.keys():
    header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 78)

for driver in pred_df.index:
    team = df.loc[df["Driver"] == driver, "Team"].values[0]
    row  = f"  {driver:<6}{team:<20}"
    for name in models.keys():
        pos = pred_df.loc[driver, name]
        row += f"  P{pos:<{col_w-2}}"
    best_pos = pred_df.loc[driver, "BestModelPos"]
    row += f"  P{best_pos:<{col_w-2}}"
    print(row)

# ── Podium from best model ────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1 = best_order.index[0]
p2 = best_order.index[1]
p3 = best_order.index[2]

t1 = df.loc[df["Driver"] == p1, "Team"].values[0]
t2 = df.loc[df["Driver"] == p2, "Team"].values[0]
t3 = df.loc[df["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*55}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather: {df['Temperature'].iloc[0]:.1f}°C | Rain: {df['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*55}")

# ── Consensus podium ──────────────────────────────────────────────────────────
print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 35)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models.keys():
        drivers_at_pos = pred_df[pred_df[name] == pos].index.tolist()
        for d in drivers_at_pos:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*55}")


   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX
   All models compared
  DRV   Team                         Ridge    ElasticNet  RandomForest GradientBoost       XGBoost          ★ Best
  ──────────────────────────────────────────────────────────────────────────────
  RUS   Mercedes              P1             P2             P1             P2             P1             P1           
  ANT   Mercedes              P2             P1             P1             P1             P1             P2           
  LEC   Ferrari               P3             P3             P1             P3             P3             P3           
  HAM   Ferrari               P4             P4             P4             P4             P4             P4           
  GAS   Alpine                P5             P6             P5             P5             P5             P5           
  BEA   Haas                  P6             P5             P6             P6             P6             P6           
  NOR   Mc

Pulling requests from Fast f1

In [4]:
import sys
print(sys.executable)

/usr/bin/python3


In [5]:
#import sys
!{sys.executable} -m pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00


In [7]:
import fastf1

In [8]:
# ── Load Miami qualifying session ─────────────────────────────────────────────
session = fastf1.get_session(2026, "Miami", "Q")
session.load(telemetry=False, weather=True, messages=False)
 
laps = session.laps
 
def td_to_s(td):
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()
 
records = []
for driver in laps["Driver"].unique():
    drv_all = laps.pick_drivers(driver).pick_wo_box()
    if drv_all.empty:
        continue
    best_s1  = drv_all["Sector1Time"].dropna().min()
    best_s2  = drv_all["Sector2Time"].dropna().min()
    best_s3  = drv_all["Sector3Time"].dropna().min()
    best_lap = drv_all["LapTime"].dropna().min()
    records.append({
        "Driver":    driver,
        "S1":        td_to_s(best_s1),
        "S2":        td_to_s(best_s2),
        "S3":        td_to_s(best_s3),
        "BestLap_s": td_to_s(best_lap),
    })
 
quali_df = pd.DataFrame(records)
quali_df["UltimateLap_s"] = (quali_df["S1"] + quali_df["S2"] + quali_df["S3"]).round(3)
 
pole_time = quali_df["BestLap_s"].min()
quali_df["MiamiGapFromPole_s"]   = (quali_df["BestLap_s"] - pole_time).round(3)
quali_df["MiamiGapFromPolePct"]  = (quali_df["MiamiGapFromPole_s"] / pole_time * 100).round(4)
 
quali_df = quali_df.sort_values("BestLap_s").reset_index(drop=True)
quali_df["MiamiGrid"] = quali_df.index + 1

req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for Miami Grand Prix - Qualifying [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Miami Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data

In [13]:
quali_df.head(10)

,Driver,S1,S2,S3,BestLap_s,UltimateLap_s,MiamiGapFromPole_s,MiamiGapFromPolePct,MiamiGrid,RainProbability,Temperature,DriverPoints,DriverPointsNorm
0,ANT,29.743,33.431,24.624,87.798,87.798,0.000,0.0000,1,0.4,28.0,72,1.0000
1,VER,30.072,33.172,24.646,87.964,87.890,0.166,0.1891,2,0.4,28.0,12,0.1667
2,LEC,29.861,33.339,24.744,88.143,87.944,0.345,0.3929,3,0.4,28.0,49,0.6806
3,NOR,30.053,33.258,24.814,88.183,88.125,0.385,0.4385,4,0.4,28.0,25,0.3472
4,RUS,29.806,33.500,24.661,88.197,87.967,0.399,0.4545,5,0.4,28.0,63,0.8750
5,HAM,29.907,33.402,24.743,88.319,88.052,0.521,0.5934,6,0.4,28.0,41,0.5694
6,PIA,29.945,33.248,24.877,88.332,88.070,0.534,0.6082,7,0.4,28.0,21,0.2917
7,COL,30.247,33.624,24.829,88.762,88.700,0.964,1.0980,8,0.4,28.0,1,0.0139
8,HAD,30.272,33.398,24.849,88.789,88.519,0.991,1.1287,9,0.4,28.0,4,0.0556
9,GAS,30.194,33.640,24.804,88.810,88.638,1.012,1.1526,10,0.4,28.0,15,0.2083


Adding weather conditions

In [10]:
rain_probability = 0.40   # 40% precipitation probability (race day)
temperature      = 28.0   # ~28°C peak, ~25°C at earlier 13:00 start

quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

In [11]:
# ── Driver standings after Japan GP (R3) ─────────────────────────────────────
driver_points = {
    "ANT": 72, "RUS": 63, "LEC": 49, "HAM": 41,
    "NOR": 25, "PIA": 21, "BEA": 17, "GAS": 15,
    "VER": 12, "LAW": 10, "LIN":  4, "HAD":  4,
    "BOR":  2, "SAI":  2, "COL":  1, "OCO":  1,
    "ALB":  0, "PER":  0, "STR":  0, "ALO":  0,
    "BOT":  0, "HUL":  0,
}

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)

In [12]:
print(quali_df[["Driver", "S1", "S2", "S3", "UltimateLap_s",
                "BestLap_s", "MiamiGrid", "MiamiGapFromPole_s",
                "MiamiGapFromPolePct", "RainProbability", "Temperature",
                "DriverPointsNorm"]])

   Driver      S1      S2      S3  UltimateLap_s  BestLap_s  MiamiGrid  \
0     ANT  29.743  33.431  24.624         87.798     87.798          1   
1     VER  30.072  33.172  24.646         87.890     87.964          2   
2     LEC  29.861  33.339  24.744         87.944     88.143          3   
3     NOR  30.053  33.258  24.814         88.125     88.183          4   
4     RUS  29.806  33.500  24.661         87.967     88.197          5   
5     HAM  29.907  33.402  24.743         88.052     88.319          6   
6     PIA  29.945  33.248  24.877         88.070     88.332          7   
7     COL  30.247  33.624  24.829         88.700     88.762          8   
8     HAD  30.272  33.398  24.849         88.519     88.789          9   
9     GAS  30.194  33.640  24.804         88.638     88.810         10   
10    BEA  30.728  33.723  24.843         89.294     89.340         11   
11    HUL  30.216  33.875  25.134         89.225     89.439         12   
12    LAW  30.530  33.810  24.998     

### Addding rolling features from Autralian, Chinese and Japan races

In [15]:
# ── Finish positions ──────────────────────────────────────────────────────────
aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}

china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

# Japan GP final classification (confirmed, The Race / PlanetF1 / ESPN)
japan_finish = {
    "ANT": 1,  "PIA": 2,  "LEC": 3,  "RUS": 4,
    "NOR": 5,  "HAM": 6,  "GAS": 7,  "VER": 8,
    "LAW": 9,  "OCO": 10, "HUL": 11, "HAD": 12,
    "BOR": 13, "LIN": 14, "SAI": 15, "COL": 16,
    "PER": 17, "ALO": 18, "BOT": 19, "ALB": 20,
    "STR": 21, "BEA": 22,   # STR DNF, BEA DNF → ranked last
}

# ── Grid / qualifying positions ───────────────────────────────────────────────
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

# Japan qualifying grid (confirmed, The Race / RacingNews365)
japan_grid = {
    "ANT": 1,  "RUS": 2,  "PIA": 3,  "LEC": 4,  "NOR": 5,
    "HAM": 6,  "GAS": 7,  "HAD": 8,  "BOR": 9,  "LIN": 10,
    "VER": 11, "OCO": 12, "HUL": 13, "LAW": 14, "COL": 15,
    "SAI": 16, "ALB": 17, "BEA": 18, "PER": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

# ── Constructor standings after Japan (R3) ────────────────────────────────────
# Source: pitdebrief.com / racingnews365.com
team_points = {
    "Mercedes":     135,
    "Ferrari":       90,
    "McLaren":       46,
    "Haas":          18,
    "Red Bull":      16,
    "Alpine":        16,
    "Racing Bulls":  15,
    "Audi":           2,
    "Williams":       1,
    "Cadillac":       0,
    "Aston Martin":   0,
}

driver_to_team = {
    "RUS": "Mercedes",     "ANT": "Mercedes",
    "HAM": "Ferrari",      "LEC": "Ferrari",
    "NOR": "McLaren",      "PIA": "McLaren",
    "VER": "Red Bull",     "HAD": "Red Bull",
    "BEA": "Haas",         "OCO": "Haas",
    "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi",         "BOR": "Audi",
    "GAS": "Alpine",       "COL": "Alpine",
    "SAI": "Williams",     "ALB": "Williams",
    "PER": "Cadillac",     "BOT": "Cadillac",
    "ALO": "Aston Martin", "STR": "Aston Martin",
}

# ── Build 3-race rolling features ─────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)  # 22

records = []
for drv in drivers:
    aus_fp  = aus_finish[drv];   chn_fp  = china_finish[drv]; jpn_fp  = japan_finish[drv]
    aus_gp  = aus_grid[drv];     chn_gp  = china_grid[drv];   jpn_gp  = japan_grid[drv]

    avg_finish_pos  = round((aus_fp + chn_fp + jpn_fp) / 3, 2)
    avg_finish_norm = round(((aus_fp-1)/(N-1) + (chn_fp-1)/(N-1) + (jpn_fp-1)/(N-1)) / 3, 4)

    avg_grid_pos  = round((aus_gp + chn_gp + jpn_gp) / 3, 2)
    avg_grid_norm = round(((aus_gp-1)/(N-1) + (chn_gp-1)/(N-1) + (jpn_gp-1)/(N-1)) / 3, 4)

    # DNF: STR & BEA in Japan, VER & ALO & STR (DNF) in China, PIA & HUL in Aus
    dnf_aus = 1 if aus_fp >= 18 else 0
    dnf_chn = 1 if chn_fp >= 16 else 0
    dnf_jpn = 1 if jpn_fp >= 21 else 0   # STR=21, BEA=22 DNF
    dnf_rate = round((dnf_aus + dnf_chn + dnf_jpn) / 3, 3)

    records.append({
        "Driver":        drv,
        "Team":          driver_to_team[drv],
        "AusFinish":     aus_fp, "ChinaFinish": chn_fp, "JapanFinish": jpn_fp,
        "AvgFinishPos":  avg_finish_pos,
        "AvgFinishNorm": avg_finish_norm,
        "AusGrid":       aus_gp, "ChinaGrid": chn_gp, "JapanGrid_prev": jpn_gp,
        "AvgGridPos":    avg_grid_pos,
        "AvgGridNorm":   avg_grid_norm,
        "DNF_rate":      dnf_rate,
    })

rolling_df = pd.DataFrame(records)

max_team_pts   = max(team_points.values())
max_driver_pts = max(driver_points.values())

rolling_df["TeamScore"] = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)

rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)

print("\n" + "=" * 90)
print("  ROLLING FEATURES FOR MIAMI GP PREDICTION")
print("  Based on: Australia (R1) + China (R2) + Japan (R3)")
print("=" * 90)
print(f"\n{'DRV':<6}{'Team':<16}{'AUS':<5}{'CHN':<5}{'JPN':<5}{'AvgFin':<9}"
      f"{'AvgFinNorm':<12}{'AvgGrid':<9}{'AvgGridNorm':<13}"
      f"{'DNF%':<7}{'TeamScore':<11}{'DrvPtsNorm'}")
print("─" * 105)
for _, r in rolling_df.sort_values("AvgFinishPos").iterrows():
    print(f"{r['Driver']:<6}{r['Team']:<16}{r['AusFinish']:<5}{r['ChinaFinish']:<5}"
          f"{r['JapanFinish']:<5}{r['AvgFinishPos']:<9}{r['AvgFinishNorm']:<12}"
          f"{r['AvgGridPos']:<9}{r['AvgGridNorm']:<13}{r['DNF_rate']:<7}"
          f"{r['TeamScore']:<11}{r['DriverPointsNorm']}")

# ── Load Miami Sprint race pace (Miami is a Sprint weekend — no FP2) ──────────
# The Sprint race gives the best available race-pace signal for the GP.
# We use ALL classified Sprint laps (typically ~19 laps) filtered to:
#   - IsAccurate == True
#   - Within 107% of the driver's own median (removes SC/VSC laps)
#   - Any compound (Sprint tyre choice is less constrained)
session_spr = fastf1.get_session(2026, "Miami", "Sprint")
session_spr.load(telemetry=False, weather=False, messages=False)

spr_laps = session_spr.laps.copy()
spr_laps["LapTime_s"] = spr_laps["LapTime"].apply(td_to_s)
spr_laps = spr_laps[spr_laps["IsAccurate"] == True]

def within_107(group):
    med = group["LapTime_s"].median()
    return group[group["LapTime_s"] <= med * 1.07]

spr_laps = spr_laps.groupby("Driver", group_keys=False).apply(within_107)

fp2_records = []
for driver, grp in spr_laps.groupby("Driver"):
    fp2_records.append({
        "Driver":           driver,
        "FP2_MedianPace_s": round(grp["LapTime_s"].median(), 3),
        "FP2_MeanPace_s":   round(grp["LapTime_s"].mean(),   3),
        "FP2_BestPace_s":   round(grp["LapTime_s"].min(),    3),
        "FP2_LapCount":     len(grp),
        "FP2_Compounds":    str(grp["Compound"].value_counts().to_dict()),
    })

fp2_df = pd.DataFrame(fp2_records).sort_values("FP2_MedianPace_s").reset_index(drop=True)
ref_pace = fp2_df["FP2_MedianPace_s"].min()
fp2_df["FP2_GapToFastest_s"] = (fp2_df["FP2_MedianPace_s"] - ref_pace).round(3)

print("\nSprint race pace (proxy for GP long-run pace):")
print(fp2_df[["Driver", "FP2_MedianPace_s", "FP2_GapToFastest_s",
              "FP2_LapCount", "FP2_Compounds"]])

# ── Merge all data ────────────────────────────────────────────────────────────
df = quali_df.merge(
    rolling_df[["Driver", "AvgFinishNorm", "AvgGridNorm",
                "DNF_rate", "TeamScore", "Team"]],
    on="Driver", how="left"
)

df_final = df.merge(
    fp2_df[["Driver", "FP2_GapToFastest_s"]],
    on="Driver", how="left"
)


core           INFO 	Loading data for Miami Grand Prix - Sprint [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Miami Grand Prix - Sprint [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...



  ROLLING FEATURES FOR MIAMI GP PREDICTION
  Based on: Australia (R1) + China (R2) + Japan (R3)

DRV   Team            AUS  CHN  JPN  AvgFin   AvgFinNorm  AvgGrid  AvgGridNorm  DNF%   TeamScore  DrvPtsNorm
─────────────────────────────────────────────────────────────────────────────────────────────────────────
ANT   Mercedes        2    1    1    1.33     0.0159      1.33     0.0159       0.0    1.0        1.0
RUS   Mercedes        1    2    4    2.33     0.0635      1.67     0.0317       0.0    1.0        0.875
LEC   Ferrari         3    4    3    3.33     0.1111      4.0      0.1429       0.0    0.6667     0.6806
HAM   Ferrari         4    3    6    4.33     0.1587      5.33     0.2063       0.0    0.6667     0.5694
GAS   Alpine          10   6    7    7.67     0.3175      9.33     0.3968       0.0    0.1185     0.2083
NOR   McLaren         5    19   5    9.67     0.4127      5.67     0.2222       0.333  0.3407     0.3472
LAW   Racing Bulls    13   7    9    9.67     0.4127      12.

req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F


Sprint race pace (proxy for GP long-run pace):
   Driver  FP2_MedianPace_s  FP2_GapToFastest_s  FP2_LapCount   FP2_Compounds
0     NOR            92.219               0.000            18  {'MEDIUM': 18}
1     LEC            92.425               0.206            18  {'MEDIUM': 18}
2     PIA            92.437               0.218            18  {'MEDIUM': 18}
3     ANT            92.571               0.352            18  {'MEDIUM': 18}
4     VER            92.623               0.404            18  {'MEDIUM': 18}
5     RUS            92.692               0.473            18  {'MEDIUM': 18}
6     HAM            92.978               0.759            18  {'MEDIUM': 18}
7     GAS            93.665               1.446            18  {'MEDIUM': 18}
8     HAD            93.727               1.508            18  {'MEDIUM': 18}
9     COL            93.950               1.731            18  {'MEDIUM': 18}
10    ALB            94.094               1.875            16  {'MEDIUM': 16}
11    BOR       

/tmp/ipykernel_13475/2830228681.py:163: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spr_laps = spr_laps.groupby("Driver", group_keys=False).apply(within_107)


### Feature Engineering

In [16]:
feature_cols = [
    "UltimateLap_s",        # Miami quali: best S1+S2+S3
    "MiamiGapFromPole_s",   # Miami quali: gap to pole
    "MiamiGrid",            # Miami qualifying position
    "FP2_GapToFastest_s",   # FP2 long-run race pace
    "TeamScore",            # Constructor points normalised (after Japan R3)
    "DriverPointsNorm",     # Driver points normalised (after Japan R3)
    "AvgFinishNorm",        # Rolling avg finish (Aus + China + Japan)
    "AvgGridNorm",          # Rolling avg grid (Aus + China + Japan)
    "DNF_rate",             # Reliability signal (3 races)
    "RainProbability",      # Weather: 40% race day rain
    "Temperature",          # Weather: ~28°C
]

# ── Target variable: rolling normalised race score ────────────────────────────
df_final["RaceScore"] = (
    0.6 * df_final["AvgFinishNorm"] +
    0.4 * df_final["AvgGridNorm"]
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


### Modelling

In [18]:
# ── Multi-model LOO-CV comparison ─────────────────────────────────────────────
models = {
    "Ridge":         Ridge(alpha=1.0),
    "ElasticNet":    ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest":  RandomForestRegressor(
                         n_estimators=100, max_depth=3,
                         min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                         n_estimators=100, learning_rate=0.05,
                         max_depth=2, subsample=0.8, random_state=42),
    "XGBoost":       XGBRegressor(
                         n_estimators=100, learning_rate=0.05,
                         max_depth=2, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=2.0,
                         random_state=42),
}

loo     = LeaveOneOut()
results = {}

for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        pipe.fit(X_imputed[train_idx], y.iloc[train_idx])
        errors.append(abs(pipe.predict(X_imputed[test_idx])[0] - y.iloc[test_idx].iloc[0]))
    results[name] = {"MAE": round(np.mean(errors), 4), "StdDev": round(np.std(errors), 4)}

print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")

# ── Best model final prediction ───────────────────────────────────────────────
best_name  = min(results, key=lambda x: results[x]["MAE"])
best_model = models[best_name]
final_pipe = (Pipeline([("scaler", StandardScaler()), ("model", best_model)])
              if best_name in ["Ridge", "ElasticNet"]
              else Pipeline([("model", best_model)]))
final_pipe.fit(X_imputed, y)
df_final["PredictedScore"] = final_pipe.predict(X_imputed)

final = df_final.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 70)
print("   PREDICTED FINISHING ORDER — 2026 MIAMI GRAND PRIX")
print("   Rolling target: Aus + China + Japan avg finish + avg grid")
print("=" * 70)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'Score'}")
print("  " + "─" * 45)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}{row['PredictedScore']:.4f}")

podium = final.head(3)
print(f"\n  {'='*50}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:   {best_name}")
print(f"  LOO MAE:      {results[best_name]['MAE']:.4f}")
print(f"  LOO Std Dev:  {results[best_name]['StdDev']:.4f}")
print(f"  Weather:      {df_final['Temperature'].iloc[0]:.1f}°C | Rain: {df_final['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*50}")

# ── Feature importances ───────────────────────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<28} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients:")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        bar = "█" * int(abs(coef) * 20)
        print(f"  {feat:<28} {coef:+.4f}  {bar}")

# ── All-models side-by-side comparison ───────────────────────────────────────
all_predictions = {}
for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    pipe.fit(X_imputed, y)
    ranked = pd.Series(pipe.predict(X_imputed),
                       index=df_final["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df_final["Driver"]
).rank(method="min").astype(int)
pred_df = pred_df.sort_values("BestModelPos")

print("\n" + "=" * 95)
print("   PREDICTED FINISHING ORDER — 2026 MIAMI GRAND PRIX  (all models)")
print("=" * 95)
col_w  = 14
header = f"  {'DRV':<6}{'Team':<22}"
for name in models: header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 88)
for driver in pred_df.index:
    team = df_final.loc[df_final["Driver"] == driver, "Team"].values[0]
    row  = f"  {driver:<6}{team:<22}"
    for name in models:
        row += f"  P{pred_df.loc[driver, name]:<{col_w-2}}"
    row += f"  P{pred_df.loc[driver, 'BestModelPos']:<{col_w-2}}"
    print(row)

# ── Consensus podium ──────────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1, p2, p3 = best_order.index[0], best_order.index[1], best_order.index[2]
t1 = df_final.loc[df_final["Driver"] == p1, "Team"].values[0]
t2 = df_final.loc[df_final["Driver"] == p2, "Team"].values[0]
t3 = df_final.loc[df_final["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*58}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather: {df_final['Temperature'].iloc[0]:.1f}°C | Rain: {df_final['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*58}")

print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 40)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models:
        for d in pred_df[pred_df[name] == pos].index:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*58}")



Model               LOO MAE  Std Dev  Verdict
───────────────────────────────────────────────────────
Ridge                0.0196   0.0185   ← best
GradientBoost        0.0398   0.0318  
ElasticNet           0.0499   0.0403  
XGBoost              0.0601   0.0561  
RandomForest         0.0717   0.0569  

   PREDICTED FINISHING ORDER — 2026 MIAMI GRAND PRIX
   Rolling target: Aus + China + Japan avg finish + avg grid
  P   DRV   Team                  Score
  ─────────────────────────────────────────────
  1   ANT   Mercedes              0.0084
  2   RUS   Mercedes              0.0496
  3   LEC   Ferrari               0.1260
  4   HAM   Ferrari               0.1862
  5   NOR   McLaren               0.3352
  6   GAS   Alpine                0.3732
  7   PIA   McLaren               0.4553
  8   LAW   Racing Bulls          0.4744
  9   HAD   Red Bull              0.4783
  10  VER   Red Bull              0.4830
  11  LIN   Racing Bulls          0.4983
  12  OCO   Haas                  0.5308


### Pulling requests from Fast F1 API

In [ ]:
import sys
print(sys.executable)

/usr/bin/python3


In [ ]:
import sys
!{sys.executable} -m pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00


In [ ]:
import fastf1
import pandas as pd

import numpy as np
import os

In [ ]:
# ── Load the session ──────────────────────────────────────────────────────────
session = fastf1.get_session(2026, "Japan", "Q")
session.load(telemetry=False, weather=True, messages=False)

laps = session.laps


req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Japanese Grand Prix - Qualifying [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_statu

In [ ]:
# ── Best individual sector times per driver (ultimate lap) ────────────────────
# FastF1 stores sector times as timedeltas — convert to seconds
def td_to_s(td):
    """Convert timedelta (or NaT) to float seconds."""
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()

records = []
for driver in laps["Driver"].unique():
    drv_all = laps.pick_drivers(driver).pick_wo_box()
    
    if drv_all.empty:
        continue  # skip drivers with no valid laps

    best_s1 = drv_all["Sector1Time"].dropna().min()
    best_s2 = drv_all["Sector2Time"].dropna().min()
    best_s3 = drv_all["Sector3Time"].dropna().min()
    best_lap = drv_all["LapTime"].dropna().min()  # ← get best lap directly

    records.append({
        "Driver":    driver,
        "S1":        td_to_s(best_s1),
        "S2":        td_to_s(best_s2),
        "S3":        td_to_s(best_s3),
        "BestLap_s": td_to_s(best_lap),
    })

quali_df = pd.DataFrame(records)

# ── Ultimate lap = S1 + S2 + S3 ───────────────────────────────────────────────
quali_df["UltimateLap_s"] = (
    quali_df["S1"] + quali_df["S2"] + quali_df["S3"]
).round(3)

# ── Gap from pole (use best Q3 / overall best lap) ───────────────────────────
pole_time = quali_df["BestLap_s"].min()
quali_df["JapanGapFromPole_s"] = (quali_df["BestLap_s"] - pole_time).round(3)

# Percentage gap — more comparable across circuits than raw seconds
quali_df["JapanGapFromPolePct"] = (
    quali_df["JapanGapFromPole_s"] / pole_time * 100
).round(4)

# ── Grid position (sort by best lap) ─────────────────────────────────────────
quali_df = quali_df.sort_values("BestLap_s").reset_index(drop=True)
quali_df["JapanGrid"] = quali_df.index + 1

# Adding The tremperature + rainfall probability on that day
# Japan GP weather — sourced manually (e.g. Weather.com / AccuWeather)
rain_probability = 0.10   # replace with actual figure
temperature      = 18.0   # replace with actual figure

quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

# Adding the Driver points from the 2 races(Australia and Chinese GPs)
driver_points = {
    "RUS": 44, "LEC": 38, "ANT": 35, "NOR": 32, "HAM": 28,
    "PIA": 22, "VER": 18, "BEA": 14, "LIN": 12, "BOR": 10,
    "GAS": 8,  "LAW": 6,  "OCO": 4,  "ALB": 4,  "HAD": 2,
    "COL": 1,  "SAI": 0,  "ALO": 0,  "BOT": 0,
    "STR": 0,  "PER": 0,  "HUL": 0,
}

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)


print(quali_df[["Driver", "S1", "S2", "S3", "UltimateLap_s",
                "BestLap_s", "JapanGrid", "JapanGapFromPole_s",
                "JapanGapFromPolePct", "RainProbability", "Temperature", "DriverPointsNorm"]])

quali_df.to_csv("japan_quali.csv", index=False)
print("\nSaved → japan_quali.csv")

   Driver      S1      S2      S3  UltimateLap_s  BestLap_s  JapanGrid  \
0     ANT  31.827  39.398  17.464         88.689     88.778          1   
1     RUS  31.782  39.607  17.616         89.005     89.076          2   
2     PIA  31.954  39.557  17.577         89.088     89.132          3   
3     LEC  31.655  39.855  17.668         89.178     89.303          4   
4     NOR  32.049  39.716  17.627         89.392     89.409          5   
5     HAM  31.762  39.933  17.625         89.320     89.567          6   
6     GAS  32.143  40.029  17.519         89.691     89.691          7   
7     HAD  32.164  40.094  17.608         89.866     89.978          8   
8     BOR  32.288  40.149  17.492         89.929     89.990          9   
9     LIN  32.548  40.043  17.501         90.092     90.109         10   
10    VER  32.369  40.196  17.565         90.130     90.262         11   
11    OCO  32.481  40.048  17.700         90.229     90.309         12   
12    HUL  32.400  40.181  17.564     

## Building rolling features from AUstralia and Chinese GPs

In [ ]:
# Let us build the rolling features from the average of the Australian and ChineseGPs grid positions and final race positions
aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}
 


In [ ]:
# China finish positions (confirmed classification)
# DNF: VER=16, ALO=17, STR=18 | DNS: NOR=19, BOR=20, ALB=21, PIA=22
china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

In [ ]:
# Australia grid positions (confirmed qualifying)
# VER P20 (crashed Q1), SAI P21 & STR P22 (power unit issues in quali)
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

In [ ]:
# China grid positions (confirmed qualifying)
# ALB P18 but started from pit lane (parc fermé modification)
china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

In [ ]:
# CONSTRUCTOR STANDINGS after China (Round 2)
# Source: formula1.com / total-motorsport.com
# ─────────────────────────────────────────────────────────────────────────────
team_points = {
    "Mercedes":     98,
    "Ferrari":      67,
    "McLaren":      18,
    "Haas":         17,
    "Red Bull":     12,
    "Racing Bulls": 12,
    "Alpine":       10,
    "Audi":          2,
    "Williams":      2,
    "Cadillac":      0,
    "Aston Martin":  0,
}

In [ ]:
# DRIVER STANDINGS after China (Round 2)
# Source: autohebdof1.com / total-motorsport.com
# ─────────────────────────────────────────────────────────────────────────────
driver_points = {
    "RUS": 51, "ANT": 47, "LEC": 34, "HAM": 33,
    "BEA": 17, "NOR": 15, "GAS":  9, "VER":  8,
    "LAW":  8, "LIN":  4, "HAD":  4, "PIA":  3,
    "SAI":  2, "BOR":  2, "COL":  1, "OCO":  0,
    "HUL":  0, "ALB":  0, "BOT":  0, "PER":  0,
    "ALO":  0, "STR":  0,
}

In [ ]:
# DRIVER → TEAM MAPPING (2026 season)
# ─────────────────────────────────────────────────────────────────────────────
driver_to_team = {
    "RUS": "Mercedes",    "ANT": "Mercedes",
    "HAM": "Ferrari",     "LEC": "Ferrari",
    "NOR": "McLaren",     "PIA": "McLaren",
    "VER": "Red Bull",    "HAD": "Red Bull",
    "BEA": "Haas",        "OCO": "Haas",
    "LAW": "Racing Bulls","LIN": "Racing Bulls",
    "HUL": "Audi",        "BOR": "Audi",
    "GAS": "Alpine",      "COL": "Alpine",
    "SAI": "Williams",    "ALB": "Williams",
    "PER": "Cadillac",    "BOT": "Cadillac",
    "ALO": "Aston Martin","STR": "Aston Martin",
}

In [ ]:
# BUILD ROLLING FEATURES
# ─────────────────────────────────────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)  # 22
 
records = []
for drv in drivers:
    aus_fp  = aus_finish[drv]
    chn_fp  = china_finish[drv]
    aus_gp  = aus_grid[drv]
    chn_gp  = china_grid[drv]
 
    # Average finish position (dense rank, so DNFs are ranked last)
    avg_finish_pos = round((aus_fp + chn_fp) / 2, 2)
 
    # Normalised average finish (0 = best, 1 = worst)
    avg_finish_norm = round(((aus_fp - 1) / (N - 1) + (chn_fp - 1) / (N - 1)) / 2, 4)
 
    # Average grid position
    avg_grid_pos = round((aus_gp + chn_gp) / 2, 2)
 
    # Normalised average grid (0 = pole, 1 = last)
    avg_grid_norm = round(((aus_gp - 1) / (N - 1) + (chn_gp - 1) / (N - 1)) / 2, 4)
 
    # DNF rate (1 race out of 2 if DNF/DNS in either)
    dnf_aus = 1 if aus_fp >= 18 else 0   # positions 18-22 = DNF or DNS
    dnf_chn = 1 if chn_fp >= 16 else 0   # VER(16), ALO(17), STR(18) DNF; NOR+ DNS
    dnf_rate = round((dnf_aus + dnf_chn) / 2, 2)
 
    records.append({
        "Driver":          drv,
        "Team":            driver_to_team[drv],
        "AusFinish":       aus_fp,
        "ChinaFinish":     chn_fp,
        "AvgFinishPos":    avg_finish_pos,
        "AvgFinishNorm":   avg_finish_norm,
        "AusGrid":         aus_gp,
        "ChinaGrid_prev":  chn_gp,
        "AvgGridPos":      avg_grid_pos,
        "AvgGridNorm":     avg_grid_norm,
        "DNF_rate":        dnf_rate,
    })
 
rolling_df = pd.DataFrame(records)

In [ ]:
# NORMALISE TEAM & DRIVER STANDINGS
# ─────────────────────────────────────────────────────────────────────────────
max_team_pts   = max(team_points.values())
max_driver_pts = max(driver_points.values())
 
rolling_df["TeamScore"]       = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)
 
rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)
 

In [ ]:
# PRINT SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("  ROLLING FEATURES FOR JAPAN GP PREDICTION")
print("  Based on: Australia (R1) + China (R2)")
print("=" * 80)
print(f"\n{'DRV':<6}{'Team':<16}{'AUS':<5}{'CHN':<5}{'AvgFin':<9}"
      f"{'AvgFinNorm':<12}{'AvgGrid':<9}{'AvgGridNorm':<13}"
      f"{'DNF%':<7}{'TeamScore':<11}{'DrvPtsNorm'}")
print("─" * 100)
for _, r in rolling_df.sort_values("AvgFinishPos").iterrows():
    print(f"{r['Driver']:<6}{r['Team']:<16}{r['AusFinish']:<5}{r['ChinaFinish']:<5}"
          f"{r['AvgFinishPos']:<9}{r['AvgFinishNorm']:<12}{r['AvgGridPos']:<9}"
          f"{r['AvgGridNorm']:<13}{r['DNF_rate']:<7}{r['TeamScore']:<11}"
          f"{r['DriverPointsNorm']}")
 
rolling_df.to_csv("rolling_features_aus_china.csv", index=False)
print("\nSaved → rolling_features_aus_china.csv")
 


  ROLLING FEATURES FOR JAPAN GP PREDICTION
  Based on: Australia (R1) + China (R2)

DRV   Team            AUS  CHN  AvgFin   AvgFinNorm  AvgGrid  AvgGridNorm  DNF%   TeamScore  DrvPtsNorm
────────────────────────────────────────────────────────────────────────────────────────────────────
RUS   Mercedes        1    2    1.5      0.0238      1.5      0.0238       0.0    1.0        1.0
ANT   Mercedes        2    1    1.5      0.0238      1.5      0.0238       0.0    1.0        0.9216
HAM   Ferrari         4    3    3.5      0.119       5.0      0.1905       0.0    0.6837     0.6471
LEC   Ferrari         3    4    3.5      0.119       4.0      0.1429       0.0    0.6837     0.6667
BEA   Haas            7    5    6.0      0.2381      13.0     0.5714       0.0    0.1735     0.3333
GAS   Alpine          10   6    8.0      0.3333      10.5     0.4524       0.0    0.102      0.1765
LAW   Racing Bulls    13   7    10.0     0.4286      11.0     0.4762       0.0    0.1224     0.1569
LIN   Racing 

In [ ]:
# HOW TO MERGE INTO YOUR MAIN DATAFRAME
# ─────────────────────────────────────────────────────────────────────────────
# After building your Japan quali df, merge like this:
#
df = quali_df.merge(
    rolling_df[["Driver", "AvgFinishNorm", "AvgGridNorm",
                "DNF_rate", "TeamScore", "Team"]],
    on="Driver", how="left"
   )

In [ ]:
df.head()

,Driver,S1,S2,S3,BestLap_s,UltimateLap_s,JapanGapFromPole_s,JapanGapFromPolePct,JapanGrid,RainProbability,Temperature,DriverPoints,DriverPointsNorm,AvgFinishNorm,AvgGridNorm,DNF_rate,TeamScore,Team
0,ANT,31.827,39.398,17.464,88.778,88.689,0.000,0.0000,1,0.1,18.0,35,0.7955,0.0238,0.0238,0.0,1.0000,Mercedes
1,RUS,31.782,39.607,17.616,89.076,89.005,0.298,0.3357,2,0.1,18.0,44,1.0000,0.0238,0.0238,0.0,1.0000,Mercedes
2,PIA,31.954,39.557,17.577,89.132,89.088,0.354,0.3987,3,0.1,18.0,22,0.5000,0.9762,0.1905,1.0,0.1837,McLaren
3,LEC,31.655,39.855,17.668,89.303,89.178,0.525,0.5914,4,0.1,18.0,38,0.8636,0.1190,0.1429,0.0,0.6837,Ferrari
4,NOR,32.049,39.716,17.627,89.409,89.392,0.631,0.7108,5,0.1,18.0,32,0.7273,0.5238,0.2381,0.5,0.1837,McLaren


 ## Analysing FP2 pace for Japan

In [ ]:
# ── Load FP2 ──────────────────────────────────────────────────────────────────
session = fastf1.get_session(2026, "Japan", "FP2")
session.load(telemetry=False, weather=False, messages=False)

laps = session.laps.copy()

# ── Helper: convert timedelta to seconds ─────────────────────────────────────
def td_to_s(td):
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()

laps["LapTime_s"] = laps["LapTime"].apply(td_to_s)

# ── Filter criteria ───────────────────────────────────────────────────────────
# Keep only representative race-sim laps:
#   - Not an in/out lap (IsAccurate flag or LapNumber > 1 in stint)
#   - Lap time within 110% of session median (removes outliers, VSC laps)
#   - Compound is MEDIUM or HARD (race compounds — not quali soft runs)
#   - Stint lap number >= 3 (skip initial warm-up laps)

laps = laps[laps["IsAccurate"] == True].copy()  # FastF1 flags dodgy laps

COMPOUNDS = ["MEDIUM", "HARD"]
laps = laps[laps["Compound"].isin(COMPOUNDS)]

# Stint lap number — laps within a stint counted from 1
laps["StintLapNum"] = laps.groupby(["Driver", "Stint"]).cumcount() + 1
laps = laps[laps["StintLapNum"] >= 3]

# Remove outliers: keep within 107% of driver's own median
def within_107(group):
    med = group["LapTime_s"].median()
    return group[group["LapTime_s"] <= med * 1.07]

laps = laps.groupby("Driver", group_keys=False).apply(within_107)

# ── Aggregate per driver ──────────────────────────────────────────────────────
records = []
for driver, grp in laps.groupby("Driver"):
    med_pace   = grp["LapTime_s"].median()
    mean_pace  = grp["LapTime_s"].mean()
    best_pace  = grp["LapTime_s"].min()
    lap_count  = len(grp)

    # Compound breakdown (useful for context)
    compound_counts = grp["Compound"].value_counts().to_dict()

    records.append({
        "Driver":           driver,
        "FP2_MedianPace_s": round(med_pace, 3),
        "FP2_MeanPace_s":   round(mean_pace, 3),
        "FP2_BestPace_s":   round(best_pace, 3),
        "FP2_LapCount":     lap_count,
        "FP2_Compounds":    str(compound_counts),
    })

fp2_df = pd.DataFrame(records).sort_values("FP2_MedianPace_s").reset_index(drop=True)

# ── Gap from fastest driver in FP2 ───────────────────────────────────────────
ref_pace = fp2_df["FP2_MedianPace_s"].min()
fp2_df["FP2_GapToFastest_s"] = (fp2_df["FP2_MedianPace_s"] - ref_pace).round(3)

print(fp2_df[["Driver", "FP2_MedianPace_s", "FP2_GapToFastest_s",
              "FP2_LapCount", "FP2_Compounds"]])

fp2_df.to_csv("japan_fp2_pace.csv", index=False)
print("\nSaved → japan_fp2_pace.csv")


core           INFO 	Loading data for Japanese Grand Prix - Practice 2 [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Japanese Grand Prix - Practice 2 [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for se

   Driver  FP2_MedianPace_s  FP2_GapToFastest_s  FP2_LapCount   FP2_Compounds
0     ANT            94.496               0.000             7   {'MEDIUM': 7}
1     STR            95.042               0.546             2   {'MEDIUM': 2}
2     RUS            95.079               0.583             6   {'MEDIUM': 6}
3     LEC            95.243               0.747             8   {'MEDIUM': 8}
4     PIA            95.808               1.312             8   {'MEDIUM': 8}
5     GAS            95.919               1.423            10    {'HARD': 10}
6     OCO            95.958               1.462            12  {'MEDIUM': 12}
7     HUL            96.146               1.650             6     {'HARD': 6}
8     HAD            96.161               1.665             9     {'HARD': 9}
9     BEA            96.170               1.674            10  {'MEDIUM': 10}
10    VER            96.299               1.803            12    {'HARD': 12}
11    HAM            96.424               1.928             3   

/tmp/ipykernel_9900/275636439.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps["StintLapNum"] = laps.groupby(["Driver", "Stint"]).cumcount() + 1
/tmp/ipykernel_9900/275636439.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  laps = laps.groupby("Driver", group_keys=False).apply(within_107)


In [ ]:
df_final = df.merge(
    fp2_df[["Driver", "FP2_GapToFastest_s"]],
    on="Driver", how="left"
   )

In [ ]:
df_final.head()

,Driver,S1,S2,S3,BestLap_s,UltimateLap_s,JapanGapFromPole_s,JapanGapFromPolePct,JapanGrid,RainProbability,Temperature,DriverPoints,DriverPointsNorm,AvgFinishNorm,AvgGridNorm,DNF_rate,TeamScore,Team,FP2_GapToFastest_s
0,ANT,31.827,39.398,17.464,88.778,88.689,0.000,0.0000,1,0.1,18.0,35,0.7955,0.0238,0.0238,0.0,1.0000,Mercedes,0.000
1,RUS,31.782,39.607,17.616,89.076,89.005,0.298,0.3357,2,0.1,18.0,44,1.0000,0.0238,0.0238,0.0,1.0000,Mercedes,0.583
2,PIA,31.954,39.557,17.577,89.132,89.088,0.354,0.3987,3,0.1,18.0,22,0.5000,0.9762,0.1905,1.0,0.1837,McLaren,1.312
3,LEC,31.655,39.855,17.668,89.303,89.178,0.525,0.5914,4,0.1,18.0,38,0.8636,0.1190,0.1429,0.0,0.6837,Ferrari,0.747
4,NOR,32.049,39.716,17.627,89.409,89.392,0.631,0.7108,5,0.1,18.0,32,0.7273,0.5238,0.2381,0.5,0.1837,McLaren,17.719


## Feature Engineering

In [ ]:
feature_cols = [
"UltimateLap_s",        # Japan quali: best S1+S2+S3
"JapanGapFromPole_s",   # Japan quali: gap to pole
"JapanGrid",            # Japan qualifying position
"FP2_GapToFastest_s",   # FP2 long-run race pace
"TeamScore",            # Constructor points normalised (after China)
"DriverPointsNorm",     # Driver points normalised (after China)
"AvgFinishNorm",        # Rolling avg finish (Aus + China)
"AvgGridNorm",          # Rolling avg grid (Aus + China)
"DNF_rate",             # Reliability signal
"RainProbability",      # Weather (manual)
"Temperature",          # Weather (manual)
]

In [ ]:
# Lets build our target variable
# ── Target: rolling normalised race score from Aus + China ───────────────
n_drivers = len(df_final)

# Normalise average finish position (0 = best, 1 = worst)
avg_finish_norm = df_final["AvgFinishNorm"]  # already in your df

# Normalise average grid position (0 = best, 1 = worst)  
avg_grid_norm = df_final["AvgGridNorm"]      # already in your df

# Composite target — weight finish more than grid (60/40)
df_final["RaceScore"] = (
    0.6 * avg_finish_norm +
    0.4 * avg_grid_norm
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

In [ ]:
import sklearn
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

In [ ]:
# Filling any missing values with median
from sklearn.impute import SimpleImputer
imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

### Modelling

In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# ── Compare multiple models with LOO-CV ──────────────────────────────────────
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error

In [ ]:
models = {
    "Ridge":        Ridge(alpha=1.0),
    "ElasticNet":   ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest": RandomForestRegressor(
                        n_estimators=100, max_depth=3,
                        min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                        n_estimators=100, learning_rate=0.05,
                        max_depth=2, subsample=0.8, random_state=42),
    "XGBoost":      XGBRegressor(
                        n_estimators=100, learning_rate=0.05,
                        max_depth=2, subsample=0.8,
                        colsample_bytree=0.8, reg_lambda=2.0,
                        random_state=42),
}

loo = LeaveOneOut()
results = {}

for name, mdl in models.items():
    # Ridge and ElasticNet benefit from scaling
    if name in ["Ridge", "ElasticNet"]:
        pipe = Pipeline([("scaler", StandardScaler()), ("model", mdl)])
    else:
        pipe = Pipeline([("model", mdl)])

    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        X_tr, X_te = X_imputed[train_idx], X_imputed[test_idx]
        y_tr, y_te = y.iloc[train_idx],    y.iloc[test_idx]
        pipe.fit(X_tr, y_tr)
        errors.append(abs(pipe.predict(X_te)[0] - y_te.iloc[0]))

    results[name] = {
        "MAE":    round(np.mean(errors), 4),
        "StdDev": round(np.std(errors), 4),
    }


In [ ]:
# ── Print comparison ──────────────────────────────────────────────────────────
print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")


Model               LOO MAE  Std Dev  Verdict
───────────────────────────────────────────────────────
Ridge                0.0245   0.0257   ← best
GradientBoost        0.0524   0.0615  
ElasticNet           0.0584   0.0374  
XGBoost              0.0687   0.0740  
RandomForest         0.0806   0.0710  


In [ ]:
# ── Use the best model for final prediction ───────────────────────────────────
best_name = min(results, key=lambda x: results[x]["MAE"])
print(f"\nUsing: {best_name}")

best_model = models[best_name]
if best_name in ["Ridge", "ElasticNet"]:
    final_pipe = Pipeline([("scaler", StandardScaler()), ("model", best_model)])
else:
    final_pipe = Pipeline([("model", best_model)])

final_pipe.fit(X_imputed, y)
df["PredictedScore"] = final_pipe.predict(X_imputed)


Using: Ridge


In [ ]:
final_pipe.fit(X_imputed, y)
df["PredictedScore"] = final_pipe.predict(X_imputed)

# ── Predicted finishing order ─────────────────────────────────────────────────
final = df.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 68)
print("   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX")
print("   Target: Aus + China avg finish + avg grid (rolling)")
print("=" * 68)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'Score'}")
print("  " + "─" * 40)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}{row['PredictedScore']:.4f}")

podium = final.head(3)
print(f"\n  {'='*45}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:        {best_name}")
print(f"  LOO MAE:           {results[best_name]['MAE']:.4f}")
print(f"  LOO Std Dev:       {results[best_name]['StdDev']:.4f}")
print(f"  Weather:           {df['Temperature'].iloc[0]:.1f}°C | Rain: {df['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*45}")

# ── Feature importances (tree models only) ────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<28} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients (higher = more influence):")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        bar = "█" * int(abs(coef) * 20)
        print(f"  {feat:<28} {coef:+.4f}  {bar}")


   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX
   Target: Aus + China avg finish + avg grid (rolling)
  P   DRV   Team                  Score
  ────────────────────────────────────────
  1   RUS   Mercedes              0.0203
  2   ANT   Mercedes              0.0216
  3   LEC   Ferrari               0.1321
  4   HAM   Ferrari               0.1572
  5   GAS   Alpine                0.3976
  6   BEA   Haas                  0.4036
  7   NOR   McLaren               0.4130
  8   LAW   Racing Bulls          0.4638
  9   LIN   Racing Bulls          0.4646
  10  HAD   Red Bull              0.4983
  11  OCO   Haas                  0.5250
  12  VER   Red Bull              0.5311
  13  COL   Alpine                0.5479
  14  BOR   Audi                  0.6049
  15  PIA   McLaren               0.6339
  16  SAI   Williams              0.6345
  17  HUL   Audi                  0.6427
  18  ALB   Williams              0.7086
  19  PER   Cadillac              0.7512
  20  BOT   Cadillac      

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Get predictions from all models ──────────────────────────────────────────
all_predictions = {}

for name, mdl in models.items():
    if name in ["Ridge", "ElasticNet"]:
        pipe = Pipeline([("scaler", StandardScaler()), ("model", mdl)])
    else:
        pipe = Pipeline([("model", mdl)])

    pipe.fit(X_imputed, y)
    preds = pipe.predict(X_imputed)
    ranked = pd.Series(preds, index=df["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df["Driver"]
).rank(method="min").astype(int)

pred_df = pred_df.sort_values("BestModelPos")

# ── Print all model predictions side by side ──────────────────────────────────
print("\n" + "=" * 85)
print("   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX")
print("   All models compared")
print("=" * 85)

col_w = 14
header = f"  {'DRV':<6}{'Team':<20}"
for name in models.keys():
    header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 78)

for driver in pred_df.index:
    team = df.loc[df["Driver"] == driver, "Team"].values[0]
    row  = f"  {driver:<6}{team:<20}"
    for name in models.keys():
        pos = pred_df.loc[driver, name]
        row += f"  P{pos:<{col_w-2}}"
    best_pos = pred_df.loc[driver, "BestModelPos"]
    row += f"  P{best_pos:<{col_w-2}}"
    print(row)

# ── Podium from best model ────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1 = best_order.index[0]
p2 = best_order.index[1]
p3 = best_order.index[2]

t1 = df.loc[df["Driver"] == p1, "Team"].values[0]
t2 = df.loc[df["Driver"] == p2, "Team"].values[0]
t3 = df.loc[df["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*55}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather: {df['Temperature'].iloc[0]:.1f}°C | Rain: {df['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*55}")

# ── Consensus podium ──────────────────────────────────────────────────────────
print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 35)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models.keys():
        drivers_at_pos = pred_df[pred_df[name] == pos].index.tolist()
        for d in drivers_at_pos:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*55}")


   PREDICTED FINISHING ORDER — 2026 JAPANESE GRAND PRIX
   All models compared
  DRV   Team                         Ridge    ElasticNet  RandomForest GradientBoost       XGBoost          ★ Best
  ──────────────────────────────────────────────────────────────────────────────
  RUS   Mercedes              P1             P2             P1             P2             P1             P1           
  ANT   Mercedes              P2             P1             P1             P1             P1             P2           
  LEC   Ferrari               P3             P3             P1             P3             P3             P3           
  HAM   Ferrari               P4             P4             P4             P4             P4             P4           
  GAS   Alpine                P5             P6             P5             P5             P5             P5           
  BEA   Haas                  P6             P5             P6             P6             P6             P6           
  NOR   Mc